# Database-level EDA: selected development and holdout databases

This notebook describes the tasks and schemas before agent evaluation. It contains no agent success metrics. Structural difficulty is derived from the ground-truth SQL: **easy** means a single-table task without joins or nesting; **non-nested complex** means relational/joined SQL without nesting; and **nested complex** means a CTE, subquery, set operation, procedural SQL, or dependent multi-object Management operation.

Databases: Cybermarket, Gaming and Museum (development); Archeology and Cross-DB (holdout). Each database contributes 15 tasks.

In [ ]:
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sqlglot
from sqlglot import exp

from sql_difficulty import add_structural_difficulty, DIFFICULTY_ORDER

ROOT = Path.cwd()
if not (ROOT / 'splits').exists():
    raise FileNotFoundError('Run this notebook from the LiveSQLBench-Agent directory.')

TRAIN_DATABASES = ['cybermarket', 'gaming', 'museum']
TEST_DATABASES = ['archeology', 'cross_db']
DATABASE_ORDER = TRAIN_DATABASES + TEST_DATABASES
DATASET_ROOT = ROOT / 'livesqlbench-base-lite'
OUTPUT_DIR = ROOT / 'results' / 'database_eda'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')

def read_jsonl(path):
    with open(path) as handle:
        return [json.loads(line) for line in handle if line.strip()]

train_rows = read_jsonl(ROOT / 'splits' / 'lite_3db_train.jsonl')
test_rows = [
    row for row in read_jsonl(ROOT / 'splits' / 'lite_7db_test.jsonl')
    if row['selected_database'] in TEST_DATABASES
]
tasks = pd.DataFrame(train_rows + test_rows).rename(columns={'selected_database': 'database'})
tasks['split'] = np.where(tasks['database'].isin(TRAIN_DATABASES), 'development', 'holdout')
tasks['database'] = pd.Categorical(tasks['database'], DATABASE_ORDER, ordered=True)
counts = tasks.groupby(['split', 'database'], observed=True).size()
assert len(tasks) == 75, f'Expected 75 tasks, found {len(tasks)}'
assert all(counts.get(('development', db), 0) == 15 for db in TRAIN_DATABASES)
assert all(counts.get(('holdout', db), 0) == 15 for db in TEST_DATABASES)
display(counts.rename('tasks').reset_index())

In [ ]:
def sql_items(value):
    if isinstance(value, list):
        return [str(item) for item in value if str(item).strip()]
    return [str(value)] if value is not None and str(value).strip() else []

def parse_sql(value):
    trees = []
    for statement in sql_items(value):
        try:
            trees.extend(tree for tree in sqlglot.parse(statement, dialect='postgres') if tree)
        except Exception:
            pass
    return trees

def canonical_join_edges(trees):
    edges = set()
    for tree in trees:
        ctes = {c.alias_or_name.lower() for c in tree.find_all(exp.CTE) if c.alias_or_name}
        aliases = {}
        for table in tree.find_all(exp.Table):
            name = (table.name or '').lower()
            if not name or name in ctes:
                continue
            aliases[name] = name
            if table.alias_or_name:
                aliases[table.alias_or_name.lower()] = name
        for join in tree.find_all(exp.Join):
            condition = join.args.get('on')
            if condition is None:
                continue
            for equality in condition.find_all(exp.EQ):
                left, right = equality.left, equality.right
                if not isinstance(left, exp.Column) or not isinstance(right, exp.Column):
                    continue
                if not left.table or not right.table:
                    continue
                ltable = aliases.get(left.table.lower(), left.table.lower())
                rtable = aliases.get(right.table.lower(), right.table.lower())
                edges.add(tuple(sorted((f'{ltable}.{left.name.lower()}', f'{rtable}.{right.name.lower()}'))))
    return edges

def sql_features(value):
    trees = parse_sql(value)
    sql_text = '\n'.join(sql_items(value))
    cte_names = {c.alias_or_name.lower() for tree in trees for c in tree.find_all(exp.CTE) if c.alias_or_name}
    tables = {
        table.name.lower() for tree in trees for table in tree.find_all(exp.Table)
        if table.name and table.name.lower() not in cte_names
    }
    columns = {column.sql(dialect='postgres').lower() for tree in trees for column in tree.find_all(exp.Column)}
    joins = [join for tree in trees for join in tree.find_all(exp.Join)]
    return {
        'unique_table_count': len(tables),
        'unique_column_count': len(columns),
        'join_count': len(joins),
        'join_edges': canonical_join_edges(trees),
        'has_join': bool(joins),
        'has_subquery': any(tree.find(exp.Subquery) is not None for tree in trees),
        'has_cte': any(tree.find(exp.CTE) is not None for tree in trees),
        'has_set_operation': any(tree.find(exp.Union, exp.Intersect, exp.Except) is not None for tree in trees),
        'has_aggregate': any(tree.find(exp.AggFunc) is not None for tree in trees),
        'has_window': any(tree.find(exp.Window) is not None for tree in trees),
        'has_group_by': any(tree.find(exp.Group) is not None for tree in trees),
        'has_order_by': any(tree.find(exp.Order) is not None for tree in trees),
        'has_distinct': any(tree.find(exp.Distinct) is not None for tree in trees),
        'sql_statement_count': max(len(trees), len([s for s in sql_text.split(';') if s.strip()])),
    }

def schema_features(database):
    schema_path = DATASET_ROOT / database / f'{database}_schema.txt'
    text = schema_path.read_text()
    blocks = re.findall(r'CREATE\s+TABLE\s+"?([A-Za-z_]\w*)"?\s*\((.*?)\);', text, re.I | re.S)
    columns = []
    fk_edges = set()
    for table, body in blocks:
        table = table.lower()
        for line in body.splitlines():
            clean = line.strip().rstrip(',')
            if not clean or re.match(r'(PRIMARY|FOREIGN|UNIQUE|CHECK|CONSTRAINT)\b', clean, re.I):
                continue
            match = re.match(r'"?([A-Za-z_]\w*)"?\s+', clean)
            if match:
                columns.append(f'{table}.{match.group(1).lower()}')
        for local, parent, remote in re.findall(
            r'FOREIGN\s+KEY\s*\("?([A-Za-z_]\w*)"?\)\s+REFERENCES\s+"?([A-Za-z_]\w*)"?\s*\("?([A-Za-z_]\w*)"?\)',
            body, re.I
        ):
            fk_edges.add(tuple(sorted((f'{table}.{local.lower()}', f'{parent.lower()}.{remote.lower()}'))))
    kb_path = DATASET_ROOT / database / f'{database}_kb.jsonl'
    kb_count = sum(1 for line in kb_path.read_text().splitlines() if line.strip())
    return {
        'schema_table_count': len(blocks),
        'schema_column_count': len(columns),
        'schema_foreign_key_count': len(fk_edges),
        'available_kb_count': kb_count,
        'foreign_key_edges': fk_edges,
    }

task_features = tasks['sol_sql'].apply(sql_features).apply(pd.Series)
analysis = pd.concat([tasks.reset_index(drop=True), task_features], axis=1)
analysis['external_knowledge'] = analysis['external_knowledge'].apply(lambda x: x if isinstance(x, list) else [])
analysis['requires_kb'] = analysis['external_knowledge'].str.len().gt(0)
analysis['required_kb_count'] = analysis['external_knowledge'].str.len()

schema_by_db = {db: schema_features(db) for db in DATABASE_ORDER}
analysis['foreign_key_join_count'] = analysis.apply(
    lambda row: len(row['join_edges'] & schema_by_db[str(row['database'])]['foreign_key_edges']), axis=1
)
analysis['non_fk_join_count'] = analysis['join_count'] - analysis['foreign_key_join_count']
analysis = add_structural_difficulty(analysis)
analysis.head()

In [ ]:
task_type = pd.crosstab(analysis['database'], analysis['category']).reindex(DATABASE_ORDER, fill_value=0)
difficulty = pd.crosstab(analysis['database'], analysis['structural_difficulty']).reindex(
    index=DATABASE_ORDER, columns=DIFFICULTY_ORDER, fill_value=0
)
print('Task type counts')
display(task_type)
print('Structurally derived difficulty counts')
display(difficulty)

feature_columns = [
    'has_join', 'has_subquery', 'has_cte', 'has_set_operation', 'has_aggregate',
    'has_window', 'has_group_by', 'has_order_by', 'has_distinct', 'requires_kb',
]
feature_rates = analysis.groupby('database', observed=True)[feature_columns].mean().reindex(DATABASE_ORDER)
print('Feature prevalence (% of tasks)')
display(feature_rates.style.format('{:.1%}'))

task_averages = analysis.groupby('database', observed=True)[[
    'unique_table_count', 'unique_column_count', 'join_count',
    'foreign_key_join_count', 'non_fk_join_count', 'required_kb_count',
    'sql_statement_count',
]].mean().reindex(DATABASE_ORDER)
task_averages.columns = [f'avg_{column}' for column in task_averages.columns]

schema_summary = pd.DataFrame.from_dict(schema_by_db, orient='index').drop(columns='foreign_key_edges')
schema_summary.index.name = 'database'
schema_summary = schema_summary.reindex(DATABASE_ORDER)
schema_summary['avg_columns_per_table'] = (
    schema_summary['schema_column_count'] / schema_summary['schema_table_count']
)
print('Database schema and KB catalogue size')
display(schema_summary.round(2))
print('Average ground-truth task complexity')
display(task_averages.round(2))

In [ ]:
database_summary = pd.concat([
    analysis.groupby('database', observed=True).size().rename('tasks'),
    task_type.add_prefix('task_'),
    difficulty.add_prefix('difficulty_'),
    feature_rates.add_suffix('_rate'),
    task_averages,
    schema_summary,
], axis=1).reindex(DATABASE_ORDER)
database_summary.insert(0, 'split', ['development'] * 3 + ['holdout'] * 2)
database_summary.to_csv(OUTPUT_DIR / 'database_summary.csv', index_label='database')
analysis.drop(columns=['join_edges']).to_csv(OUTPUT_DIR / 'task_features.csv', index=False)
display(database_summary.round(3))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(17, 12))
task_type.plot(kind='bar', stacked=True, ax=axes[0, 0], color=['#F28E2B', '#4E79A7'])
axes[0, 0].set_title('Task type by database')
axes[0, 0].set_ylabel('Tasks')
axes[0, 0].tick_params(axis='x', rotation=25)

difficulty.plot(kind='bar', stacked=True, ax=axes[0, 1], color=['#59A14F', '#F28E2B', '#E15759'])
axes[0, 1].set_title('Structural difficulty by database')
axes[0, 1].set_ylabel('Tasks')
axes[0, 1].tick_params(axis='x', rotation=25)

heat = axes[1, 0].imshow(feature_rates.T.values, cmap='Blues', vmin=0, vmax=1, aspect='auto')
axes[1, 0].set_xticks(range(len(feature_rates.index)), feature_rates.index, rotation=25)
axes[1, 0].set_yticks(range(len(feature_rates.columns)), feature_rates.columns)
for row_index in range(feature_rates.shape[1]):
    for column_index in range(feature_rates.shape[0]):
        value = feature_rates.iloc[column_index, row_index]
        axes[1, 0].text(column_index, row_index, f'{value:.0%}', ha='center', va='center', fontsize=8)
fig.colorbar(heat, ax=axes[1, 0], fraction=0.046, pad=0.04)
axes[1, 0].set_title('Ground-truth SQL feature prevalence')
axes[1, 0].set_xlabel('Database')
axes[1, 0].set_ylabel('Feature')

task_averages[[
    'avg_unique_table_count', 'avg_join_count', 'avg_foreign_key_join_count',
    'avg_non_fk_join_count', 'avg_required_kb_count'
]].plot(kind='bar', ax=axes[1, 1])
axes[1, 1].set_title('Average task-level complexity')
axes[1, 1].set_ylabel('Average per task')
axes[1, 1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'database_eda_overview.png', dpi=180, bbox_inches='tight')
plt.show()

## Interpretation guidance

- `schema_foreign_key_count` describes the database schema; `avg_foreign_key_join_count` describes how many declared-FK joins are used by an average task.
- `requires_kb_rate` is the percentage of tasks whose ground-truth metadata lists at least one external-knowledge entry.
- SQL feature rates are not mutually exclusive: one task may contain joins, aggregation, a CTE and a window function.
- Structural difficulty is derived only from the reference SQL and is not shown to the agent.
- Parser fallbacks may undercount constructs inside opaque PL/pgSQL bodies; procedural Management tasks are nevertheless classified through regex-based Management features.